In [3]:
import os
os.chdir("../")

In [87]:
import pandas as pd
from reportlab.lib import colors
from reportlab.platypus import *
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
from datetime import datetime
from pathlib import Path

# Report

In [92]:
import pandas as pd
from reportlab.lib import colors
from reportlab.platypus import *
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
from datetime import datetime
from pathlib import Path

# ======================
# CONFIG
# ======================
output_path = Path("reports/tod/truck_tod_report.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)

# ======================
# LOAD DATA
# ======================
df = pd.read_csv("data/interim/observed_data/observed_dataset_merged.csv")

# ======================
# FILTER
# ======================
remove = (df['source'] == 'bata_2023') & (df["truck_type_1"] == "struck")
df = df[~remove]

# ======================
# PROCESS
# ======================
pt = df.pivot_table(index="truck_type_1", columns="tod", values="volume", aggfunc="sum")
pct = pt.div(pt.sum(axis=1), axis=0)

row_order = ['vstruck', 'struck', 'mtruck', 'ctruck']
col_order = ['EA', 'AM', 'MD', 'PM', 'EV']
pct = pct.loc[row_order, col_order]
pct = pct.round(6)
pct.to_csv("data/processed/tod_distributions.csv")

tm16 = pd.DataFrame({
    'EA':[0.0235,0.0765,0.0665,0.1430],
    'AM':[0.0000,0.2440,0.2930,0.2320],
    'MD':[0.6080,0.3710,0.3935,0.3315],
    'PM':[0.1980,0.2180,0.1730,0.1750],
    'EV':[0.1705,0.0905,0.0740,0.1185]
}, index=row_order)

diff = pct - tm16

rename = {
    'vstruck': 'Very Small',
    'struck': 'Small',
    'mtruck': 'Medium',
    'ctruck': 'Large'
}

pct.index = pct.index.map(rename)
tm16.index = tm16.index.map(rename)
diff.index = diff.index.map(rename)

# ======================
# STYLES
# ======================
styles = getSampleStyleSheet()
doc = SimpleDocTemplate(str(output_path), pagesize=letter)
elements = []

# ======================
# HEADER TABLE (MEMO STYLE)
# ======================
header_data = [
    ["To:", "MTC"],
    ["From:", "WSP"],
    ["Cc:", ""],
    ["Ref:", ""],
    ["Date:", datetime.today().strftime('%Y-%m-%d')],
    ["Subject:", "Truck Time-of-Day Distribution Calibration"]
]

header_table = Table(header_data, colWidths=[80, 400])
header_table.setStyle(TableStyle([
    ('GRID', (0,0), (-1,-1), 0.5, colors.black),
    ('BACKGROUND', (0,0), (0,-1), colors.lightgrey)
]))

elements.append(header_table)
elements.append(Spacer(1, 20))

# ======================
# BODY TEXT
# ======================
elements.append(Paragraph("<b>Purpose</b>", styles['Heading3']))
elements.append(Paragraph(
    "This memo provides documentation of the newly estimated truck time-of-day distributions developed for TM1.7. "
    " A comparison with the current TM1.6 distributions is included to summarize how the updated estimates differ from the existing model assumptions.",
    styles['Normal']
))
elements.append(Spacer(1, 12))

elements.append(Paragraph("<b>Data Sources</b>", styles['Heading3']))
elements.append(Paragraph(
    "Observed data are derived from a combination of Caltrans 2018 and BATA 2023 datasets. "
    "Caltrans data provide detailed vehicle classification, while BATA contributes bridge-level counts.",
    styles['Normal']
))
elements.append(Spacer(1, 12))

# ======================
# YOUR PARAGRAPH
# ======================
elements.append(Paragraph("<b>Data Processing and Assumptions</b>", styles['Heading3']))
elements.append(Paragraph("""
<b>Exclusion of BATA Small Truck Observations</b><br/><br/>
Small truck observations from the BATA 2023 dataset are excluded from the analysis. 
These observations are based solely on the 2-axle classification, which combines both 
very small (2-axle, 4-tire) and small (2-axle, 6-tire) trucks. Because the BATA data 
do not distinguish between tire configurations, it is not possible to separate these 
two vehicle types.<br/><br/>
To avoid introducing bias into the estimation of time-of-day distributions, all small 
truck observations from the BATA 2023 source are removed. This exclusion has a limited 
impact on the overall dataset, as BATA 2023 provides counts for only seven bridges. 
The majority of observations are sourced from Caltrans data, where very small and small 
trucks can be identified and estimated separately.
""", styles['Normal']))
elements.append(Spacer(1, 20))

# ======================
# TABLE FUNCTION
# ======================
def make_table(df, title, heatmap=False):
    data = [["Truck Type"] + list(df.columns)]
    for idx, row in (df * 100).round(2).iterrows():
        data.append([idx] + [f"{v:.2f}%" for v in row])

    table = Table(data)

    style = TableStyle([
        ('GRID', (0,0), (-1,-1), 0.5, colors.black),
        ('BACKGROUND', (0,0), (-1,0), colors.grey)
    ])

    if heatmap:
        for i in range(1, len(df)+1):
            for j in range(1, len(df.columns)+1):
                val = df.iloc[i-1, j-1]
                color = (
                    colors.red if val <= -0.1 else
                    colors.orange if val <= -0.03 else
                    colors.white if val < 0.03 else
                    colors.lightgreen if val < 0.1 else
                    colors.green
                )
                style.add('BACKGROUND', (j,i),(j,i), color)

    table.setStyle(style)

    return [Paragraph(f"<b>{title}</b>", styles['Heading3']), table, Spacer(1, 20)]

# ======================
# RESULTS
# ======================
elements.append(Paragraph("<b>Results</b>", styles['Heading3']))

elements += make_table(tm16, "TM1.6 Distributions")
elements += make_table(pct, "Observed Distributions")
elements += make_table(diff, "Difference (Observed - TM1.6)", heatmap=True)

# ======================
# BUILD
# ======================
doc.build(elements)

print("✅ Memo generated:", output_path)

✅ Memo generated: reports\tod\truck_tod_report.pdf
